Anexo A, apartado A.4 · El ciclo completo: entrenar el adaptador y evaluar ensayo a ensayo.

Cuaderno con el que se calculó en Google Colab la verosimilitud por sesión de la base con los pesos
reinicializados al azar.


# Stage 7.5.2 [revisión interna] — Centaur vs RandomInit-70B sensitivity (standalone)

Standalone notebook **dedicado únicamente** al [revisión interna] sensitivity. Sin estado heredado del notebook principal de Stage 7.5.2 (que tiene Cells 1-7 para training + 3-way eval + sanity checks).

**Compute estimado**: ~70-90 min A100 80GB.
**Disk requerido**: ≥160 GB libres `/content/` (Cell 4 internal HF cache cleanup activado por default per commit `c4aaf06`).
**HF accesos**: gated `meta-llama/Llama-3.1-70B` + `marcelbinz/Llama-3.1-Centaur-70B-adapter` + `marcelbinz/Llama-3.1-RandomInit-70B`.

Comparación: Centaur (pretrained Llama-3.1-70B + Centaur adapter) vs `marcelbinz/Llama-3.1-RandomInit-70B` (untrained random-init base, NO adapter; HF inspection confirmó sin sibling `-Centaur-adapter`).

Verdict mapping: `centaur_beats_randominit__lower_bound_check` (esperado) — sensitivity, NO main verdict del 3-way Bonferroni-2.

Workflow obligatorio: **Save a copy in Drive** primero (cuenta `la cuenta de Drive del autor`) para coherencia Drive mount → output path.

## Cell 1: Clone repo + dependencies

In [ ]:
# Clone repo (privado, requiere GH_TOKEN en Colab Secrets)
from google.colab import userdata
import os

gh_token = None
try:
    gh_token = userdata.get('GH_TOKEN')
except Exception:
    pass

if not os.path.exists('/content/ai-system-lab'):
    if gh_token:
        !git clone <REPO> /content/ai-system-lab
    else:
        !git clone <REPO> /content/ai-system-lab
else:
    !cd /content/ai-system-lab && git pull origin main

assert os.path.exists('/content/ai-system-lab/tesis'), 'Clone failed — revisa GH_TOKEN o repo visibility'

# Install dependencies
# transformers<4.50 pin: newer versions materialize bf16 full weights before 4-bit quantization (OOM en 70B). May-19 3-way eval ejecutó con 4.49.x exitosamente.
!pip install -q "transformers>=4.43,<4.50" peft bitsandbytes accelerate scipy numpy

## Cell 2: HF_TOKEN to kernel env + Drive mount

In [ ]:
# HF_TOKEN kernel injection (NO subprocess; kernel env directly)
import os

if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
        if tok:
            os.environ['HF_TOKEN'] = tok
    except Exception as e:
        print(f'WARN: could not read Colab Secret HF_TOKEN: {e}')

assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN in Colab Secrets (🔑 sidebar) + toggle Notebook access ON'
print('✓ HF_TOKEN available in kernel env')

# Mount Drive (output path target)
from google.colab import drive
drive.mount('/content/drive')

# Critical: ensure correct Drive account (la cuenta de Drive del autor)
# Verify expected paths exist (sanity for account match)
paper_results = '/content/drive/MyDrive/paper_01_igt_results'
if os.path.exists(paper_results):
    print(f'✓ {paper_results} accessible — Drive account OK')
else:
    print(f'⚠ {paper_results} NOT found — verify Drive mounted to la cuenta de Drive del autor')
    print('   If wrong account: drive.mount("/content/drive", force_remount=True)')

## Cell 3: Disk diagnostic (pre-run sanity check)

[revisión interna] necesita ≥160 GB libres `/content/` para cargar secuencialmente Llama-3.1-70B base (140 GB) + Centaur adapter (415 MB) → cleanup integrado en script post-[1/3] → RandomInit-70B (140 GB).

In [ ]:
import shutil
from pathlib import Path

total, used, free = shutil.disk_usage('/content')
print(f'DISK ACTUAL: {used/1024**3:.1f} / {total/1024**3:.1f} GB used, {free/1024**3:.1f} GB free')
print()

# HF cache inventory
hf_cache = Path.home() / '.cache' / 'huggingface' / 'hub'
if hf_cache.exists():
    print(f'HF cache root: {hf_cache}')
    total_hf = 0
    for item in sorted(hf_cache.iterdir()):
        if item.is_dir():
            size_gb = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / 1024**3
            total_hf += size_gb
            print(f'  {item.name}: {size_gb:.1f} GB')
    print(f'  TOTAL HF cache: {total_hf:.1f} GB')
else:
    print('HF cache empty (fresh runtime)')

# Decision gate
if free / 1024**3 < 160:
    print()
    print(f'⚠ Less than 160 GB free. Runnning Cell 4 cleanup PRE-launch.')
else:
    print()
    print(f'✓ ≥160 GB libre — Cell 5 launch safe')

## Cell 4 (opcional): wipe HF cache PRE-launch

Ejecutar **solo si Cell 3 reportó <160 GB libres**. En fresh runtime no es necesario.

In [ ]:
import shutil
from pathlib import Path

hf_cache = Path.home() / '.cache' / 'huggingface' / 'hub'
if hf_cache.exists():
    before_gb = sum(f.stat().st_size for f in hf_cache.rglob('*') if f.is_file()) / 1024**3
    shutil.rmtree(hf_cache)
    hf_cache.mkdir(parents=True, exist_ok=True)
    print(f'✓ HF cache wiped: {before_gb:.1f} GB freed')
else:
    print('HF cache already empty')

total, used, free = shutil.disk_usage('/content')
print(f'DISK POST-CLEANUP: {used/1024**3:.1f} / {total/1024**3:.1f} GB used, {free/1024**3:.1f} GB free')

## Cell 5: [revisión interna] RandomInit eval (~70-90 min A100 80GB)

El script `eval_centaur_vs_randominit.py` (commit `c4aaf06`) tiene cleanup HF cache integrado entre [1/3] Centaur y [2/3] RandomInit (default `--clean-hf-cache-between-phases=True`).

**[revisión interna] patches aplicadas** (commit `8a53360`):
- P0: `--randominit-training-status untrained_random_base` (default; HF account inspection confirmó)
- P1: strict tokenizer identity check + CUDA mem diagnostics + cache stamping con manifest validation
- P2: IGT schema validation + provenance fields

Watch logs:
- `[1/3] NLL Centaur...` → ~40 min (load 8 min + inference 30 min)
- `[HF CACHE] cleaned models--meta-llama--Llama-3-1-70B (140.0 GB freed)` ← key transition
- `[2/3] NLL RandomInit-70B...` → ~45 min (download 10-15 min + load 5 min + inference 30 min)
- `[3/3] Paired stats + [revisión interna] verdict...` → <30 sec

In [ ]:
%cd /content/ai-system-lab/tesis/data_analyses/llm_evaluation/paper_01_igt/stage_7_5_2_lora_irrelevant_and_3way_eval/

!python eval_centaur_vs_randominit.py \
    --igt-data /content/ai-system-lab/tesis/data_analyses/llm_evaluation/paper_01_igt/manifests/igt_paper1_eval_data.json \
    --output /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2_randominit_sensitivity/

## Cell 6: Display verdict + zip backup (post-completion)

In [ ]:
# Display [revisión interna] verdict
import json
out_path = '/content/drive/MyDrive/paper_01_igt_results/stage_7_5_2_randominit_sensitivity/eval_centaur_vs_randominit_stats.json'
if os.path.exists(out_path):
    with open(out_path) as f:
        stats = json.load(f)
    print('=' * 60)
    print('[revisión interna] verdict:', stats.get('verdict_c25'))
    print('=' * 60)
    print(f"  paired Δ NLL = {stats['delta_mean_centaur_minus_randominit']:+.4f}")
    print(f"  Cohen's d (signed) = {stats['cohens_d']:+.3f}")
    print(f"  CI 95% = [{stats['ci_lo_95']:+.4f}, {stats['ci_hi_95']:+.4f}]")
    print(f"  p-value one-sided = {stats['p_value_one_sided']:.2e}")
    print(f"  RandomInit training status = {stats['randominit_training_status']}")
else:
    print(f'⚠ Stats file not found at {out_path}')

In [ ]:
# Zip outputs + download (safety backup; sync_stage_outputs.sh local also works tras Drive Desktop sync)
!zip -r /content/stage_7_5_2_randominit_sensitivity.zip /content/drive/MyDrive/paper_01_igt_results/stage_7_5_2_randominit_sensitivity/
from google.colab import files
files.download('/content/stage_7_5_2_randominit_sensitivity.zip')

## Done

Outputs en `/content/drive/MyDrive/paper_01_igt_results/stage_7_5_2_randominit_sensitivity/`:
- `eval_centaur_vs_randominit_stats.json` — stats + verdict + provenance (script commit + library versions + tokenizer fingerprints + IGT SHA256)
- `nll_centaur__meta-llama__Llama-3-1-70B__marcelbinz__Llama-3-1-Centaur-70B-adapter.json` — cached Centaur NLL (reusable via `--reuse-cached-nll`)
- `nll_randominit__marcelbinz__Llama-3-1-RandomInit-70B.json` — cached RandomInit NLL
- `cache_manifest.json` — provenance validation manifest

Tras completion, sync local + insertar resultado en [revisión interna] [revisión interna] anexo.